# L13 · Closed-Loop Evaluation and Capstone

This notebook turns one trained checkpoint into task-level evidence. The core path freezes a protocol, runs five live feedback episodes, audits every outcome and video, and reports a Wilson interval. The Capstone then compares the same seeds under one predeclared held-out condition.

A clean default run is diagnostic only: it audits local inputs and pure logic without loading a policy or starting Genesis. It never downloads or substitutes a checkpoint.

## Before you run

Use the complete course environment and a checkpoint whose training dataset identity is known. Configure inputs before starting the kernel:

- `RG101_L13_DATASET_ROOT` and `RG101_L13_REPO_ID` select the local training dataset (the audited L09 dataset is the diagnostic default).
- `RG101_L13_CHECKPOINT` must name a `pretrained_model` directory. There is deliberately no default checkpoint.
- `RG101_L13_OUTPUT_ROOT` defaults to `outputs/l13` and must remain below the configured outputs directory. Choose a fresh child directory for each formal run.
- `RG101_L13_RUN_EVAL=1` and `ROBO_GENESIS_RENDER=1` opt in to five reference episodes. `RG101_L13_RUN_CAPSTONE=1` separately opts in to the matched held-out condition.
- `RG101_L13_RUN_SWEEP=1` is an optional checkpoint-only extension and also requires `RG101_L13_SWEEP_RUN_DIR`.

The subprocess boundaries keep Genesis initialization out of the notebook kernel. A failed subprocess stops the notebook instead of being turned into a policy failure.

In [ ]:
import importlib.metadata as package_metadata
import json
import os
import platform
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.evaluation_contract import (
    EvaluationContractError,
    audit_dataset_contract,
    build_condition_spec,
    classify_condition,
    evaluate_predicate_trace,
    paired_outcome_changes,
    summarize_episode_rows,
    validate_evaluation_protocol,
    validate_evaluation_report,
    validate_output_root,
    validate_output_targets,
)
from robo_genesis.paths import DATASETS_DIR, OUTPUTS_DIR, PROJECT_ROOT
from robo_genesis.training_contract import audit_checkpoint

def env_flag(name):
    value = os.environ.get(name, '0').strip().lower()
    if value not in {'0', '1', 'false', 'true', 'no', 'yes'}:
        raise ValueError(f'{name} must be a boolean flag, got {value!r}')
    return value in {'1', 'true', 'yes'}

def configured_path(value, *, default=None):
    raw = value if value not in (None, '') else default
    if raw is None:
        return None
    path = Path(raw).expanduser()
    return (path if path.is_absolute() else PROJECT_ROOT / path).resolve()

def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return None

lesson = load_course_manifest().lesson('L13')
assert lesson.status.value == 'planned'
run_eval = env_flag('RG101_L13_RUN_EVAL')
run_capstone = env_flag('RG101_L13_RUN_CAPSTONE')
run_sweep = env_flag('RG101_L13_RUN_SWEEP')
render_value = os.environ.get('ROBO_GENESIS_RENDER', '1').strip().lower()
if render_value not in {'0', '1', 'false', 'true', 'no', 'yes'}:
    raise ValueError(f'ROBO_GENESIS_RENDER must be a boolean flag, got {render_value!r}')
render_enabled = render_value in {'1', 'true', 'yes'}
repo_id = os.environ.get('RG101_L13_REPO_ID', 'local/l09_banana_demo').strip()
dataset_root = configured_path(
    os.environ.get('RG101_L13_DATASET_ROOT'), default=DATASETS_DIR / 'l09_banana_demo'
)
checkpoint_root = configured_path(os.environ.get('RG101_L13_CHECKPOINT'))
output_root = configured_path(
    os.environ.get('RG101_L13_OUTPUT_ROOT'), default=OUTPUTS_DIR / 'l13'
)
output_root = validate_output_root(output_root, allowed_root=OUTPUTS_DIR)
policy_device = os.environ.get('RG101_L13_DEVICE', 'cuda').strip()
try:
    import torch
    torch_device_evidence = {
        'version': torch.__version__,
        'cuda_or_rocm_available': torch.cuda.is_available(),
        'device_count': torch.cuda.device_count(),
    }
except Exception as exc:
    torch_device_evidence = {'import_error': str(exc)}
runtime_summary = {
    'lesson': lesson.id,
    'status': lesson.status.value,
    'python': platform.python_version(),
    'genesis': installed_version('genesis-world'),
    'lerobot': installed_version('lerobot'),
    'torch': torch_device_evidence,
    'requested_policy_device': policy_device,
    'render_enabled': render_enabled,
    'modes': {'reference': run_eval, 'capstone': run_capstone, 'sweep': run_sweep},
    'dataset_root': str(dataset_root),
    'checkpoint_root': str(checkpoint_root) if checkpoint_root else None,
    'output_root': str(output_root),
}
print(json.dumps(runtime_summary, indent=2))

## 1. Audit the inputs before closing the loop

A checkpoint and a dataset form one contract: camera keys and sizes, state/action dimension, FPS, normalization state, training dataset identity, and any camera rename must agree. This gate reads metadata only. A missing checkpoint is a visible `SKIP`, not a request to search for or download another model.

In [ ]:
dataset_evidence = None
checkpoint_evidence = None
input_audit = []
try:
    dataset_evidence = audit_dataset_contract(dataset_root, repo_id=repo_id)
    input_audit.append({'input': 'dataset', 'status': 'PASS', 'detail': dataset_evidence['metadata_path']})
except EvaluationContractError as exc:
    input_audit.append({'input': 'dataset', 'status': 'FAIL', 'detail': str(exc)})
    if run_eval or run_capstone or run_sweep or 'RG101_L13_DATASET_ROOT' in os.environ:
        raise

if checkpoint_root is None:
    input_audit.append({
        'input': 'checkpoint',
        'status': 'SKIP',
        'detail': 'RG101_L13_CHECKPOINT is unset. No download or fallback checkpoint is attempted.',
    })
else:
    config_path = checkpoint_root / 'config.json'
    if not config_path.is_file():
        raise EvaluationContractError(f'checkpoint config does not exist: {config_path}')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    policy_type = config.get('type')
    if not isinstance(policy_type, str) or not policy_type:
        raise EvaluationContractError('checkpoint config must name a policy type')
    checkpoint_evidence = audit_checkpoint(
        checkpoint_root, expected_policy_type=policy_type
    )
    if checkpoint_evidence['dataset_repo_id'] != repo_id:
        raise EvaluationContractError(
            'checkpoint training dataset does not match RG101_L13_REPO_ID: '
            f"{checkpoint_evidence['dataset_repo_id']!r} != {repo_id!r}"
        )
    input_audit.append({'input': 'checkpoint', 'status': 'PASS', 'detail': str(checkpoint_root)})

inputs_ready = dataset_evidence is not None and checkpoint_evidence is not None
print(json.dumps(input_audit, indent=2))
print('Live-evaluation inputs ready:', inputs_ready)

## 2. Freeze the episode rule before seeing results

The course predicate is checked at policy FPS. Success requires it to remain true for `ceil(dwell_seconds × policy_fps)` consecutive checks. A final true frame at timeout is still a timeout if the streak is too short.

**Exercise:** edit `SYNTHETIC_PREDICATE_TRACE`, predict the result on paper, then run the cell and compare your prediction with the state machine.

In [ ]:
EPISODES = 5
BASE_SEED = 1000
CONTROL_FPS = 100
POLICY_FPS = float(dataset_evidence['fps']) if dataset_evidence else 5.0
DWELL_SECONDS = 0.4
MAX_SECONDS = 15.0
PICK_OBJECTS = ('011_banana',)
PLACE_TARGET = '024_bowl'
SUCCESS_TOLERANCE_M = 0.06
AVAILABLE_OBJECTS = (
    '003_cracker_box', '004_sugar_box', '005_tomato_soup_can',
    '006_mustard_bottle', '007_tuna_fish_can', '008_pudding_box',
    '009_gelatin_box', '010_potted_meat_can', '011_banana',
    '019_pitcher_base', '021_bleach_cleanser', '024_bowl',
)
reference_protocol = validate_evaluation_protocol(
    episodes=EPISODES,
    policy_fps=POLICY_FPS,
    control_fps=CONTROL_FPS,
    dwell_seconds=DWELL_SECONDS,
    max_seconds=MAX_SECONDS,
    tolerance=SUCCESS_TOLERANCE_M,
    pick_objects=PICK_OBJECTS,
    place_target=PLACE_TARGET,
    available_objects=AVAILABLE_OBJECTS,
    base_seed=BASE_SEED,
    balanced=True,
    condition_name='reference',
    render_enabled=True,
)
SYNTHETIC_PREDICATE_TRACE = [False, True, False, True]
trace_evidence = evaluate_predicate_trace(
    SYNTHETIC_PREDICATE_TRACE,
    dwell_seconds=DWELL_SECONDS,
    policy_fps=POLICY_FPS,
)
print(json.dumps({
    'dwell_frames': reference_protocol['success_rule']['dwell_frames'],
    'episode_seeds': reference_protocol['sampling']['episode_seeds'],
    'object_schedule': reference_protocol['sampling']['object_schedule'],
    'synthetic_trace': trace_evidence,
}, indent=2))

## 3. Name conditions relative to training support

The reference condition is the report's predeclared baseline. A second condition is `unseen-in-support` only when its range lies inside documented training support, `OOD` only when the ranges are disjoint, and otherwise `shifted`.

**Exercise:** set one `RG101_L13_CHANGED_FACTOR` (`friction_ratio` or `mass_ratio`) plus `RG101_L13_EVALUATION_RANGE=LO,HI`. Add `RG101_L13_TRAINING_SUPPORT=LO,HI` only when dataset provenance supports that claim. Change no second factor, and explain the resulting label before running anything.

In [ ]:
def optional_range(name):
    raw = os.environ.get(name)
    if raw in (None, ''):
        return None
    values = [float(item.strip()) for item in raw.split(',')]
    if len(values) != 2:
        raise ValueError(f'{name} must contain LO,HI')
    return values

reference_condition = build_condition_spec(
    name='reference',
    declared_label='reference',
    held_constant=('task', 'camera contract', 'episode seeds'),
)
CHANGED_FACTOR = os.environ.get('RG101_L13_CHANGED_FACTOR') or None
TRAINING_SUPPORT = optional_range('RG101_L13_TRAINING_SUPPORT')
EVALUATION_RANGE = optional_range('RG101_L13_EVALUATION_RANGE')
HELD_OUT_NAME = os.environ.get('RG101_L13_HELD_OUT_NAME', 'held-out-condition')
held_out_condition = None
if CHANGED_FACTOR is not None:
    if CHANGED_FACTOR not in {'friction_ratio', 'mass_ratio'}:
        raise ValueError('This notebook isolates friction_ratio or mass_ratio only')
    if EVALUATION_RANGE is None:
        raise ValueError('A held-out run requires RG101_L13_EVALUATION_RANGE=LO,HI')
    held_out_label = classify_condition(
        is_reference=False,
        training_support=TRAINING_SUPPORT,
        evaluation_range=EVALUATION_RANGE,
    )
    held_out_condition = build_condition_spec(
        name=HELD_OUT_NAME,
        declared_label=held_out_label,
        changed_factor=CHANGED_FACTOR,
        training_support=TRAINING_SUPPORT,
        evaluation_range=EVALUATION_RANGE,
        held_constant=('task', 'camera contract', 'episode seeds'),
    )
condition_plan = {
    'reference': reference_condition,
    'held_out': held_out_condition or 'SKIP — no single changed factor was declared',
}
print(json.dumps(condition_plan, indent=2))

## 4. Run one checkpoint under the frozen reference condition

This is the first cell that may start expensive work. With both opt-in flags enabled, it launches the policy evaluator as a separate process, captures the complete log, and requires a fresh result/video location. Five episodes are an integration smoke, not a stable performance benchmark.

In [ ]:
def evaluation_command(condition, run_root, extra_arguments=()):
    command = [
        sys.executable, '-m', 'robo_genesis.eval_policy',
        '--policy-path', str(checkpoint_root),
        '--repo-id', repo_id,
        '--dataset-root', str(dataset_root),
        '--device', policy_device,
        '--episodes', str(EPISODES),
        '--seed', str(BASE_SEED),
        '--max-seconds', str(MAX_SECONDS),
        '--dwell-seconds', str(DWELL_SECONDS),
        '--pick', *PICK_OBJECTS,
        '--place', PLACE_TARGET,
        '--tol', str(SUCCESS_TOLERANCE_M),
        '--save-video',
        '--condition-name', condition['name'],
        '--condition-label', condition['label'],
        '--video-dir', str(run_root / 'videos'),
        '--results-out', str(run_root / 'results.json'),
    ]
    for item in condition['held_constant']:
        command.extend(['--held-constant', item])
    command.extend(extra_arguments)
    return command

def run_logged_command(command, log_path):
    print('$ ' + shlex.join(command))
    process = subprocess.run(
        command, cwd=PROJECT_ROOT, capture_output=True, text=True, check=False
    )
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(process.stdout + process.stderr, encoding='utf-8')
    if process.returncode != 0:
        raise RuntimeError(
            f'evaluator exited with code {process.returncode}; inspect {log_path}'
        )
    return process

reference_root = output_root / 'reference'
reference_paths = {
    'results': reference_root / 'results.json',
    'videos': reference_root / 'videos',
    'log': reference_root / 'run.log',
}
reference_command = evaluation_command(reference_condition, reference_root)
reference_report = None
reference_status = 'NOT RUN'
if run_eval:
    if not render_enabled:
        raise RuntimeError('RG101_L13_RUN_EVAL=1 requires ROBO_GENESIS_RENDER=1')
    if not inputs_ready:
        raise RuntimeError('Reference evaluation requires audited dataset and checkpoint inputs')
    validate_output_targets(
        results_path=reference_paths['results'],
        video_dir=reference_paths['videos'],
        log_path=reference_paths['log'],
        save_video=True,
    )
    run_logged_command(reference_command, reference_paths['log'])
    reference_data = json.loads(reference_paths['results'].read_text(encoding='utf-8'))
    reference_data['outputs']['log'] = str(reference_paths['log'])
    reference_report = validate_evaluation_report(reference_data)
    reference_paths['results'].write_text(json.dumps(reference_report, indent=2), encoding='utf-8')
    reference_status = 'COMPLETED'
else:
    print('SKIP — RG101_L13_RUN_EVAL is not enabled; no evaluator subprocess was started.')
print('Reference rollout:', reference_status)

## 5. Inspect rollout evidence

The report tells us how often each outcome occurred; a selected video helps us inspect what happened in one episode. This cell requires one decodable H.264 video per episode, checks dimensions and frame counts, and chooses the first failure when one exists (otherwise the first episode).

In [ ]:
video_evidence = []
representative_episode = None
if reference_report is not None:
    import cv2
    for episode in reference_report['episodes']:
        video_value = episode.get('video')
        if not video_value:
            raise EvaluationContractError(f"episode seed {episode['seed']} has no video path")
        video_path = Path(video_value)
        if not video_path.is_file() or video_path.stat().st_size <= 0:
            raise EvaluationContractError(f'missing or empty rollout video: {video_path}')
        capture = cv2.VideoCapture(str(video_path))
        if not capture.isOpened():
            raise EvaluationContractError(f'cannot decode rollout video: {video_path}')
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        fourcc = int(capture.get(cv2.CAP_PROP_FOURCC))
        codec = ''.join(chr((fourcc >> (8 * index)) & 0xFF) for index in range(4)).strip('\x00').lower()
        capture.release()
        if width <= 0 or height <= 0 or width % 2 or height % 2 or frames <= 0:
            raise EvaluationContractError(f'invalid video geometry or frame count: {video_path}')
        if codec not in {'avc1', 'h264', 'x264'}:
            raise EvaluationContractError(f'expected H.264 video, got codec {codec!r}: {video_path}')
        video_evidence.append({
            'seed': episode['seed'], 'path': str(video_path), 'codec': codec,
            'width': width, 'height': height, 'frames': frames,
        })
    if len(video_evidence) != reference_report['summary']['episodes']:
        raise EvaluationContractError('video count does not match the episode denominator')
    representative_episode = next(
        (row for row in reference_report['episodes'] if not row['success']),
        reference_report['episodes'][0],
    )
    selection_rule = 'first failure' if not representative_episode['success'] else 'first episode'
    print(json.dumps({'videos': video_evidence, 'selection_rule': selection_rule}, indent=2))
    from IPython.display import Video, display
    display(Video(representative_episode['video'], embed=False))
else:
    print('SKIP — no reference rollout report exists, so there is no video evidence to audit.')

## 6. Recompute the result from episode rows

A percentage alone hides both the denominator and small-sample uncertainty. The versioned report is accepted only when its `k/n`, Wilson interval, termination counts, per-object counts, seeds, protocol, and condition can all be recomputed from the episode rows.

**Exercise:** after a real run, add one failed seed to `MANUAL_FAILURE_LABELS`. Record a short category and only evidence visible in that episode's video; do not infer a cause from the aggregate success rate.

In [ ]:
MANUAL_FAILURE_LABELS = {
    # 1003: {'category': 'release timing', 'visible_evidence': 'object leaves the gripper beside the bowl'},
}
result_audit_status = 'NOT RUN'
reference_summary = None
if reference_report is not None:
    validate_evaluation_report(reference_report)
    reference_summary = summarize_episode_rows(reference_report['episodes'])
    if reference_summary != reference_report['summary']:
        raise EvaluationContractError('stored summary differs from recomputed episode counts')
    failed_seeds = {row['seed'] for row in reference_report['episodes'] if not row['success']}
    for seed, annotation in MANUAL_FAILURE_LABELS.items():
        if seed not in failed_seeds:
            raise EvaluationContractError(f'manual label seed {seed} is not a failed episode')
        if set(annotation) != {'category', 'visible_evidence'} or not all(annotation.values()):
            raise EvaluationContractError(f'manual label for seed {seed} is incomplete')
    result_audit_status = 'COMPLETED'
    print(json.dumps({
        'successes_over_episodes': f"{reference_summary['successes']}/{reference_summary['episodes']}",
        'success_rate': reference_summary['success_rate'],
        'wilson_95': reference_summary['wilson_95'],
        'termination_counts': reference_summary['termination_counts'],
        'per_object': reference_summary['per_object'],
        'episode_seeds': [row['seed'] for row in reference_report['episodes']],
        'manual_failure_labels': MANUAL_FAILURE_LABELS,
    }, indent=2))
else:
    print('SKIP — counts and Wilson interval require an actual reference report.')
print('Result audit:', result_audit_status)

## 7. Compare one held-out condition with matched seeds

The Capstone condition changes one documented factor and keeps the task, camera contract, object schedule, and five episode seeds matched. Outcome flips are paired evidence about these sampled task instances; they are not, by themselves, a significance claim. If training support is missing or overlaps the evaluation range, keep the neutral `shifted` label rather than claiming OOD.

In [ ]:
def held_out_cli_arguments(condition):
    arguments = [
        '--changed-factor', condition['changed_factor'],
        '--evaluation-range', *[str(value) for value in condition['evaluation_range']],
    ]
    if condition['training_support'] is not None:
        arguments.extend([
            '--training-support', *[str(value) for value in condition['training_support']]
        ])
    return arguments

held_out_report = None
paired_changes = []
condition_comparison_status = 'NOT RUN'
if run_capstone:
    if not run_eval or reference_report is None:
        raise RuntimeError('Capstone requires a completed reference run in this kernel')
    if held_out_condition is None:
        raise RuntimeError('Declare exactly one held-out factor before enabling Capstone')
    held_out_root = output_root / held_out_condition['name']
    held_out_paths = {
        'results': held_out_root / 'results.json',
        'videos': held_out_root / 'videos',
        'log': held_out_root / 'run.log',
    }
    held_out_command = evaluation_command(
        held_out_condition, held_out_root, held_out_cli_arguments(held_out_condition)
    )
    validate_output_targets(
        results_path=held_out_paths['results'],
        video_dir=held_out_paths['videos'],
        log_path=held_out_paths['log'],
        save_video=True,
    )
    run_logged_command(held_out_command, held_out_paths['log'])
    held_out_data = json.loads(held_out_paths['results'].read_text(encoding='utf-8'))
    held_out_data['outputs']['log'] = str(held_out_paths['log'])
    held_out_report = validate_evaluation_report(held_out_data)
    held_out_paths['results'].write_text(json.dumps(held_out_report, indent=2), encoding='utf-8')
    paired_changes = paired_outcome_changes(
        reference_report['episodes'], held_out_report['episodes']
    )
    condition_comparison_status = 'COMPLETED'
    print(json.dumps({
        'reference': reference_report['summary'],
        held_out_condition['name']: held_out_report['summary'],
        'paired_outcome_changes': paired_changes,
        'interpretation': 'Matched outcomes; no significance claim is made.',
    }, indent=2))
else:
    print('SKIP — RG101_L13_RUN_CAPSTONE is not enabled; no held-out subprocess was started.')
print('Condition comparison:', condition_comparison_status)

## 8. Optional extension: compare saved checkpoints

A checkpoint sweep asks how saved states from one training run behave under the same reference protocol. Its cost is `number of checkpoints × episodes`. One checkpoint is one point, not a learning curve, and the core Capstone does not require this extension.

In [ ]:
sweep_run_root = configured_path(os.environ.get('RG101_L13_SWEEP_RUN_DIR'))
sweep_checkpoints = []
if sweep_run_root is not None:
    checkpoint_parent = sweep_run_root / 'checkpoints'
    if checkpoint_parent.is_dir():
        sweep_checkpoints = sorted(
            path.parent for path in checkpoint_parent.glob('[0-9]*/pretrained_model/config.json')
        )
sweep_budget = len(sweep_checkpoints) * EPISODES
sweep_output = output_root / 'checkpoint-sweep'
sweep_command = None
if sweep_run_root is not None:
    sweep_command = [
        sys.executable, '-m', 'robo_genesis.eval_sweep',
        '--run-dir', str(sweep_run_root),
        '--repo-id', repo_id,
        '--dataset-root', str(dataset_root),
        '--device', policy_device,
        '--episodes', str(EPISODES),
        '--seed', str(BASE_SEED),
        '--max-seconds', str(MAX_SECONDS),
        '--pick', *PICK_OBJECTS,
        '--place', PLACE_TARGET,
        '--tol', str(SUCCESS_TOLERANCE_M),
        '--output-dir', str(sweep_output),
    ]
print(json.dumps({
    'checkpoints': [str(path) for path in sweep_checkpoints],
    'episodes_per_checkpoint': EPISODES,
    'total_rollouts': sweep_budget,
    'command': shlex.join(sweep_command) if sweep_command else None,
}, indent=2))
sweep_status = 'NOT RUN'
if run_sweep:
    if not render_enabled or not inputs_ready:
        raise RuntimeError('Checkpoint sweep requires render plus audited dataset/checkpoint inputs')
    if len(sweep_checkpoints) < 2:
        raise RuntimeError('A learning-curve sweep requires at least two numeric checkpoints')
    if sweep_output.exists() and any(sweep_output.iterdir()):
        raise EvaluationContractError(f'sweep output is not empty: {sweep_output}')
    run_logged_command(sweep_command, sweep_output / 'run.log')
    sweep_status = 'COMPLETED'
else:
    print('SKIP — RG101_L13_RUN_SWEEP is not enabled; command shown for budget review only.')
print('Checkpoint sweep:', sweep_status)

## 9. Capstone: answer one question with a traceable evidence chain

The report should answer one scoped question, not merely list completed steps. It traces the conclusion back through environment, dataset, checkpoint, frozen protocols, episode rows, videos, and limitations.

**Exercise:** complete three boundary statements in the cell: what this run supports, what it cannot support, and which additional evidence you would collect next. The report is not written while any statement is blank.

In [ ]:
CAPSTONE_QUESTION = os.environ.get(
    'RG101_L13_CAPSTONE_QUESTION',
    'How did this checkpoint behave under the reference and one predeclared held-out condition?'
)
SUPPORTED_CLAIM = os.environ.get('RG101_L13_SUPPORTED_CLAIM', '')
UNSUPPORTED_CLAIM = os.environ.get('RG101_L13_UNSUPPORTED_CLAIM', '')
NEXT_EVIDENCE = os.environ.get('RG101_L13_NEXT_EVIDENCE', '')
capstone_report_status = 'NOT COMPLETED'
capstone_paths = None
if held_out_report is not None:
    boundary_statements = [SUPPORTED_CLAIM, UNSUPPORTED_CLAIM, NEXT_EVIDENCE]
    if all(statement.strip() for statement in boundary_statements):
        capstone_root = output_root / 'capstone'
        capstone_paths = {
            'json': capstone_root / 'report.json',
            'markdown': capstone_root / 'report.md',
        }
        validate_output_targets(
            results_path=capstone_paths['json'],
            video_dir=None,
            log_path=capstone_paths['markdown'],
            save_video=False,
        )
        capstone_report = {
            'schema_name': 'robo-genesis-l13-capstone-report',
            'schema_version': 1,
            'generated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
            'question': CAPSTONE_QUESTION,
            'evidence': {
                'environment': reference_report['environment'],
                'artifacts': reference_report['artifacts'],
                'reference_report': str(reference_paths['results']),
                'held_out_report': str(held_out_paths['results']),
                'reference_summary': reference_report['summary'],
                'held_out_summary': held_out_report['summary'],
                'paired_outcome_changes': paired_changes,
            },
            'conclusion_boundary': {
                'supported': SUPPORTED_CLAIM,
                'not_supported': UNSUPPORTED_CLAIM,
                'next_evidence': NEXT_EVIDENCE,
            },
            'limitations': sorted(set(reference_report['limitations'] + held_out_report['limitations'])),
        }
        reference_k = reference_report['summary']['successes']
        reference_n = reference_report['summary']['episodes']
        held_out_k = held_out_report['summary']['successes']
        held_out_n = held_out_report['summary']['episodes']
        markdown_report = (
            f"# L13 Capstone report\n\n## Question\n\n{CAPSTONE_QUESTION}\n\n"
            f"## Observed evidence\n\n- Reference: {reference_k}/{reference_n}, "
            f"95% Wilson interval {reference_report['summary']['wilson_95']}\n"
            f"- {held_out_condition['name']}: {held_out_k}/{held_out_n}, "
            f"95% Wilson interval {held_out_report['summary']['wilson_95']}\n\n"
            f"## Conclusion boundary\n\n- Supported: {SUPPORTED_CLAIM}\n"
            f"- Not supported: {UNSUPPORTED_CLAIM}\n- Next evidence: {NEXT_EVIDENCE}\n"
        )
        capstone_root.mkdir(parents=True, exist_ok=True)
        capstone_paths['json'].write_text(json.dumps(capstone_report, indent=2), encoding='utf-8')
        capstone_paths['markdown'].write_text(markdown_report, encoding='utf-8')
        capstone_report_status = 'COMPLETED'
        print(markdown_report)
    else:
        print('INCOMPLETE — fill the three conclusion-boundary statements before writing the report.')
else:
    print('SKIP — a Capstone report requires both actual condition reports.')
print('Capstone report:', capstone_report_status)

## 10. Final evidence check

The final cell keeps four statuses separate. Core completion requires an audited checkpoint, a real accelerator-selected five-episode reference run, real rendering, one valid video per episode, and a recomputed report. Capstone completion additionally requires the matched held-out report and the written evidence-chain report. A checkpoint sweep remains optional.

In [ ]:
reference_device = (
    reference_report.get('environment', {}).get('actual_policy_device', '')
    if reference_report is not None else ''
)
core_complete = bool(
    inputs_ready
    and reference_status == 'COMPLETED'
    and result_audit_status == 'COMPLETED'
    and reference_report['summary']['episodes'] == EPISODES
    and reference_report['protocol']['render_enabled']
    and reference_device.startswith('cuda')
    and len(video_evidence) == EPISODES
)
capstone_complete = bool(
    core_complete
    and condition_comparison_status == 'COMPLETED'
    and held_out_report is not None
    and len(paired_changes) == EPISODES
    and capstone_report_status == 'COMPLETED'
)
diagnostic_complete = bool(reference_protocol and trace_evidence and input_audit)
print('Diagnostic/preflight:', 'COMPLETED' if diagnostic_complete else 'NOT COMPLETED')
print('Core closed-loop evaluation: COMPLETED' if core_complete else 'Core closed-loop evaluation: NOT COMPLETED')
print('Capstone: COMPLETED' if capstone_complete else 'Capstone: NOT COMPLETED')
print('Optional checkpoint sweep:', sweep_status)